In [111]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [112]:
#importing dataset

df= pd.read_csv('yield_df.csv')
df.head()

,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


In [113]:
#deleting unwanted column

df.drop('Unnamed: 0', axis=1, inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28242 entries, 0 to 28241
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Area                           28242 non-null  str    
 1   Item                           28242 non-null  str    
 2   Year                           28242 non-null  int64  
 3   hg/ha_yield                    28242 non-null  int64  
 4   average_rain_fall_mm_per_year  28242 non-null  float64
 5   pesticides_tonnes              28242 non-null  float64
 6   avg_temp                       28242 non-null  float64
dtypes: float64(3), int64(2), str(2)
memory usage: 1.5 MB


In [114]:
df.duplicated().sum()

np.int64(2310)

In [115]:
#deleting duplicate rows

df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [116]:
types = df['average_rain_fall_mm_per_year'].apply(type)
print(types.unique())

[<class 'float'>]


In [134]:
col=['Year', 'average_rain_fall_mm_per_year','pesticides_tonnes', 'avg_temp', 'Area', 'Item', 'hg/ha_yield']
df= df[col]

In [118]:
x= df.drop('hg/ha_yield', axis=1)
y= df['hg/ha_yield']


In [119]:
x_train, x_test, y_train, y_test= train_test_split(x,y, train_size=0.8, random_state=42)

In [120]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


In [121]:
ohe= OneHotEncoder(drop='first')
scaler= StandardScaler()


In [122]:
x_train

,Year,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp,Area,Item
24234,2000,59.0,3024.11,26.55,Saudi Arabia,Sorghum
9468,2012,652.0,8002.20,18.82,Greece,Sweet potatoes
6793,2006,3240.0,98328.63,27.51,Colombia,Maize
28212,2010,657.0,3305.17,21.17,Zimbabwe,Potatoes
7358,2007,1410.0,5689.80,27.08,Dominican Republic,Sweet potatoes
...,...,...,...,...,...,...
23678,2004,854.0,16942.00,16.31,Portugal,Sweet potatoes
5960,2006,537.0,36572.75,7.85,Canada,Wheat
860,1991,534.0,17866.00,18.73,Australia,Potatoes
17223,1998,250.0,6416.14,6.94,Kazakhstan,Potatoes


In [123]:
#encoding and scaling the data

preprocessor= ColumnTransformer(
    transformers=[
        ('onehotencoder', ohe, [4,5]),
        ('standarization', scaler, [0,1,2,3])
     ],
     remainder= 'passthrough')


In [124]:
x_train_dummy= preprocessor.fit_transform(x_train)
x_test_dummy= preprocessor.transform(x_test)


In [125]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [126]:
#training the model

models={
    'lr':LinearRegression(),
    'lss': Lasso(),
    'rg':Ridge(),
    'dtr':DecisionTreeRegressor()
}

for name, mod in models.items():
    mod.fit(x_train_dummy, y_train)
    y_pred = mod.predict(x_test_dummy)

    print(f'{name} MSE: {mean_squared_error(y_test, y_pred)} Score: {r2_score(y_test, y_pred)}')


lr MSE: 1821709155.0591247 Score: 0.7486566582459786


c:\Users\rimsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:784: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.316691e+10, tolerance: 1.482e+10
  model = cd_fast.sparse_enet_coordinate_descent(


lss MSE: 1822234158.2996433 Score: 0.7485842229351403
rg MSE: 1822541952.0402904 Score: 0.7485417562729235
dtr MSE: 163239199.5843455 Score: 0.9774776968020181


In [127]:
#model selection
#we are selecting decision tree regressor since it gives the lowest mean squared error ansd the highest r2 score

dtr= DecisionTreeRegressor()
dtr.fit(x_train_dummy, y_train)
dtr.predict(x_test_dummy)

array([154330.,  15838.,  72614., ...,  52692.,   9621., 132600.],
      shape=(5187,))

In [128]:
df.head(1)

,Year,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp,Area,Item,hg/ha_yield
0,1990,1485.0,121.0,16.37,Albania,Maize,36613


In [129]:
def prediction(Year, average_rain_fall_mm_per_year, pesticides_tonnes, avg_temp, Area, Item):
    # Create an array of the input features
    features = np.array([[Year, average_rain_fall_mm_per_year, pesticides_tonnes, avg_temp, Area, Item]], dtype=object)

    # Transform the features using the preprocessor
    transformed_features = preprocessor.transform(features)

    # Make the prediction
    predicted_yield = dtr.predict(transformed_features).reshape(1, -1)
    return predicted_yield[0]

Year = 1990
average_rain_fall_mm_per_year =1485.0
pesticides_tonnes = 121.00
avg_temp = 16.37                   
Area = 'Albania'
Item = 'Maize'
result = prediction(Year, average_rain_fall_mm_per_year, pesticides_tonnes, avg_temp, Area, Item)

c:\Users\rimsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\rimsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [130]:
result

array([36613.])

In [133]:
import pickle
pickle.dump(dtr, open('dtr.pkl', 'wb'))
pickle.dump(preprocessor, open('preprocessor.pkl', 'wb'))
